In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score
from sklearn.linear_model import LogisticRegression
import optuna

# =============================================================================
# CONFIGURATION – ADJUST FOR SPEED VS ACCURACY
# =============================================================================
COMP_PATH = "/kaggle/input/competitions/march-machine-learning-mania-2026"
N_FOLDS = 5                      # Number of CV folds
RANDOM_STATE = 42
USE_GPU = True                   # Set to False if GPU not available
SPEED_MODE = "balanced"           # "fast", "balanced", or "accurate"

# Set Optuna trials based on speed mode
if SPEED_MODE == "fast":
    N_TRIALS_OPTUNA = 0           # Skip Optuna, use pre‑tuned params
    USE_OPTUNA = False
elif SPEED_MODE == "balanced":
    N_TRIALS_OPTUNA = 20          # Quick tuning
    USE_OPTUNA = True
else:  # accurate
    N_TRIALS_OPTUNA = 50
    USE_OPTUNA = True

print(f"🚀 Starting March Machine Learning Mania 2026 – Winning Pipeline")
print(f"   Speed mode: {SPEED_MODE} | Optuna trials: {N_TRIALS_OPTUNA}")
print(f"   CV folds: {N_FOLDS} | GPU: {USE_GPU}")

🚀 Starting March Machine Learning Mania 2026 – Winning Pipeline
   Speed mode: balanced | Optuna trials: 20
   CV folds: 5 | GPU: True


In [3]:
# =============================================================================
# CELL 2: HELPER FUNCTIONS (FIXED: NO CATEGORICAL DTYPES)
# =============================================================================
def engineer_seed_features(df):
    """Transform raw seed strings into numeric features."""
    df = df.copy()
    # Extract numeric seed and region letter (keep region as string, not category)
    df['Seed_Num'] = df['Seed'].str[1:3].astype(int)
    df['Seed_Region'] = df['Seed'].str[0]          # <-- kept as string, not category
    
    # Seed strength (higher = better)
    df['Seed_Strength'] = 17 - df['Seed_Num']
    
    # Non‑linear transformations
    df['Seed_Value'] = 1 / df['Seed_Num']
    df['Seed_Squared'] = df['Seed_Num'] ** 2
    df['Seed_Percentile'] = (17 - df['Seed_Num']) / 16
    return df

def get_round(daynum):
    """Convert DayNum to tournament round (approximate)."""
    if daynum < 136:
        return 0  # regular season / conference tournaments
    # Tournament days (simplified mapping)
    if daynum < 140:
        return 1  # First Four / Round of 64
    elif daynum < 145:
        return 2  # Round of 32
    elif daynum < 150:
        return 3  # Sweet 16
    elif daynum < 155:
        return 4  # Elite 8
    elif daynum < 160:
        return 5  # Final Four
    else:
        return 6  # Championship

def compute_team_stats(reg_df):
    """
    Compute per‑team per‑season statistics from regular season games.
    """
    # Add score diff for later aggregation
    reg_df = reg_df.copy()
    reg_df['Score_Diff'] = reg_df['WScore'] - reg_df['LScore']
    
    # Winner stats
    winner = reg_df.groupby(['Season', 'WTeamID']).agg(
        Games_won=('WTeamID', 'count'),
        Pts_scored_w=('WScore', 'sum'),
        Pts_allowed_w=('LScore', 'sum'),
        Margin_w=('Score_Diff', 'sum')
    ).reset_index().rename(columns={'WTeamID': 'TeamID'})
    
    # Loser stats
    loser = reg_df.groupby(['Season', 'LTeamID']).agg(
        Games_lost=('LTeamID', 'count'),
        Pts_scored_l=('LScore', 'sum'),
        Pts_allowed_l=('WScore', 'sum'),
        Margin_l=('Score_Diff', 'sum')
    ).reset_index().rename(columns={'LTeamID': 'TeamID'})
    
    # Merge
    stats = pd.merge(winner, loser, on=['Season', 'TeamID'], how='outer').fillna(0)
    
    # Totals
    stats['Games'] = stats['Games_won'] + stats['Games_lost']
    stats['Pts_scored'] = stats['Pts_scored_w'] + stats['Pts_scored_l']
    stats['Pts_allowed'] = stats['Pts_allowed_w'] + stats['Pts_allowed_l']
    stats['Margin'] = stats['Margin_w'] + stats['Margin_l']
    
    # Averages
    stats['Avg_pts_scored'] = stats['Pts_scored'] / stats['Games']
    stats['Avg_pts_allowed'] = stats['Pts_allowed'] / stats['Games']
    stats['Avg_margin'] = stats['Margin'] / stats['Games']
    stats['Win_pct'] = stats['Games_won'] / stats['Games']
    
    keep = ['Season', 'TeamID', 'Games', 'Avg_pts_scored', 'Avg_pts_allowed',
            'Avg_margin', 'Win_pct']
    return stats[keep]

In [4]:
# =============================================================================
# CELL 3: BUILD TRAINING DATASET (FIXED: ROUND AS INT, NOT CATEGORY)
# =============================================================================
def build_dataset(reg_df, seeds_df, team_stats, tourney_df=None):
    """
    Construct training data with win/loss rows and rich features.
    Uses precomputed team_stats (per season, per team).
    """
    # Add score diff for later aggregation (will NOT be used as feature)
    reg_df = reg_df.copy()
    reg_df['Score_Diff'] = reg_df['WScore'] - reg_df['LScore']
    
    # Create winner rows
    reg_df['Win'] = 1
    
    # Create loser rows
    losers = reg_df.copy()
    losers[['WTeamID', 'LTeamID']] = losers[['LTeamID', 'WTeamID']]
    losers['Win'] = 0
    df = pd.concat([reg_df, losers], ignore_index=True)
    
    # Add tournament round if available
    if tourney_df is not None:
        tourney_df = tourney_df.copy()
        tourney_df['Round'] = tourney_df['DayNum'].apply(get_round)
        tourney_df = tourney_df[['Season', 'DayNum', 'WTeamID', 'LTeamID', 'Round']]
        df = df.merge(tourney_df, on=['Season', 'DayNum', 'WTeamID', 'LTeamID'], how='left')
        df['Round'] = df['Round'].fillna(0).astype(int)   # <-- keep as int, not category
    else:
        df['Round'] = 0
    
    # Merge seed features for team1 (winner)
    seed_cols = ['Seed_Num', 'Seed_Strength', 'Seed_Value', 'Seed_Squared', 'Seed_Percentile']
    df = df.merge(seeds_df[['Season', 'TeamID'] + seed_cols],
                  left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team1_{col}' for col in seed_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    # Merge seed features for team2 (loser)
    df = df.merge(seeds_df[['Season', 'TeamID'] + seed_cols],
                  left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team2_{col}' for col in seed_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    # Merge team stats for team1
    stat_cols = ['Avg_pts_scored', 'Avg_pts_allowed', 'Avg_margin', 'Win_pct', 'Games']
    df = df.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team1_{col}' for col in stat_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    # Merge team stats for team2
    df = df.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team2_{col}' for col in stat_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    # Fill missing values (play‑in teams, etc.)
    seed_fill_map = {
        'Seed_Num': 16,
        'Seed_Strength': 1,
        'Seed_Value': 1/16,
        'Seed_Squared': 256,
        'Seed_Percentile': 1/16
    }
    for col in seed_cols:
        df[f'Team1_{col}'] = df[f'Team1_{col}'].fillna(seed_fill_map[col])
        df[f'Team2_{col}'] = df[f'Team2_{col}'].fillna(seed_fill_map[col])
    
    # Fill team stats with median
    for col in stat_cols:
        median_val = team_stats[col].median()
        df[f'Team1_{col}'] = df[f'Team1_{col}'].fillna(median_val)
        df[f'Team2_{col}'] = df[f'Team2_{col}'].fillna(median_val)
    
    # --- Feature engineering ---
    # Seed differences
    df['Seed_Num_Diff'] = df['Team1_Seed_Num'] - df['Team2_Seed_Num']
    df['Seed_Strength_Diff'] = df['Team1_Seed_Strength'] - df['Team2_Seed_Strength']
    df['Seed_Value_Diff'] = df['Team1_Seed_Value'] - df['Team2_Seed_Value']
    df['Seed_Num_Ratio'] = df['Team1_Seed_Num'] / (df['Team2_Seed_Num'] + 1)
    df['Seed_Strength_Ratio'] = df['Team1_Seed_Strength'] / (df['Team2_Seed_Strength'] + 1)
    
    # Team stat differences
    df['Avg_pts_scored_Diff'] = df['Team1_Avg_pts_scored'] - df['Team2_Avg_pts_scored']
    df['Avg_pts_allowed_Diff'] = df['Team1_Avg_pts_allowed'] - df['Team2_Avg_pts_allowed']
    df['Avg_margin_Diff'] = df['Team1_Avg_margin'] - df['Team2_Avg_margin']
    df['Win_pct_Diff'] = df['Team1_Win_pct'] - df['Team2_Win_pct']
    
    # Interactions
    df['Seed_Num_Product'] = df['Team1_Seed_Num'] * df['Team2_Seed_Num']
    df['Seed_Sum'] = df['Team1_Seed_Num'] + df['Team2_Seed_Num']
    
    # Tier indicators
    for tier, low, high in [('Elite', 1, 4), ('Contender', 5, 8), ('Mid', 9, 12), ('Low', 13, 16)]:
        df[f'Team1_Tier_{tier}'] = ((df['Team1_Seed_Num'] >= low) & (df['Team1_Seed_Num'] <= high)).astype(int)
        df[f'Team2_Tier_{tier}'] = ((df['Team2_Seed_Num'] >= low) & (df['Team2_Seed_Num'] <= high)).astype(int)
    
    df['Same_Tier_Elite'] = ((df['Team1_Tier_Elite'] == 1) & (df['Team2_Tier_Elite'] == 1)).astype(int)
    df['Same_Tier_Low'] = ((df['Team1_Tier_Low'] == 1) & (df['Team2_Tier_Low'] == 1)).astype(int)
    df['Tier_Gap'] = abs(df['Team1_Seed_Num'] // 4 - df['Team2_Seed_Num'] // 4)
    
    # Round (integer, not categorical)
    # No conversion needed
    
    # Neutral court? (all tournament games are neutral, but regular season may have location)
    if 'WLoc' in df.columns:
        df['Is_Neutral'] = (df['WLoc'] == 'N').astype(int)
    else:
        df['Is_Neutral'] = 1
    
    # Final feature list (excluding target and raw identifiers)
    feature_cols = [
        'Seed_Num_Diff', 'Seed_Strength_Diff', 'Seed_Value_Diff',
        'Seed_Num_Ratio', 'Seed_Strength_Ratio',
        'Avg_pts_scored_Diff', 'Avg_pts_allowed_Diff', 'Avg_margin_Diff', 'Win_pct_Diff',
        'Seed_Num_Product', 'Seed_Sum',
        'Same_Tier_Elite', 'Same_Tier_Low', 'Tier_Gap',
        'Is_Neutral', 'Round'
    ]
    # Ensure all exist
    feature_cols = [c for c in feature_cols if c in df.columns]
    return df[feature_cols + ['Win']]

In [5]:
# =============================================================================
# CELL 4: LOAD DATA AND COMPUTE TEAM STATISTICS
# =============================================================================
print("\n📥 Loading data...")
sample = pd.read_csv(f"{COMP_PATH}/SampleSubmissionStage1.csv")
print(f"   Submission requires {len(sample):,} rows")

# Seeds
M_seeds_raw = pd.read_csv(f"{COMP_PATH}/MNCAATourneySeeds.csv")
W_seeds_raw = pd.read_csv(f"{COMP_PATH}/WNCAATourneySeeds.csv")

# Regular season results
M_reg = pd.read_csv(f"{COMP_PATH}/MRegularSeasonCompactResults.csv")
W_reg = pd.read_csv(f"{COMP_PATH}/WRegularSeasonCompactResults.csv")

# Tournament results
try:
    M_tourney = pd.read_csv(f"{COMP_PATH}/MNCAATourneyCompactResults.csv")
    W_tourney = pd.read_csv(f"{COMP_PATH}/WNCAATourneyCompactResults.csv")
    use_tourney = True
    print("   Tournament data loaded")
except:
    use_tourney = False
    print("   Tournament data not available – using regular season only")

# Compute team stats early (keep for test)
print("\n📊 Computing team strength statistics...")
men_stats = compute_team_stats(M_reg)
women_stats = compute_team_stats(W_reg)
print(f"   Men stats: {men_stats.shape}, Women stats: {women_stats.shape}")


📥 Loading data...
   Submission requires 519,144 rows
   Tournament data loaded

📊 Computing team strength statistics...
   Men stats: (13753, 7), Women stats: (9851, 7)


In [6]:
# =============================================================================
# CELL 5: ENGINEER SEED FEATURES
# =============================================================================
print("\n🔧 Engineering seed features...")
M_seeds = engineer_seed_features(M_seeds_raw)
W_seeds = engineer_seed_features(W_seeds_raw)


🔧 Engineering seed features...


In [7]:
# =============================================================================
# CELL 6: BUILD TRAINING DATASETS
# =============================================================================
print("\n📊 Building men's training dataset...")
M_train = build_dataset(M_reg, M_seeds, men_stats, M_tourney if use_tourney else None)
print(f"   Men: {M_train.shape}")

print("\n📊 Building women's training dataset...")
W_train = build_dataset(W_reg, W_seeds, women_stats, W_tourney if use_tourney else None)
print(f"   Women: {W_train.shape}")

feature_cols = [c for c in M_train.columns if c != 'Win']
print(f"\n🧩 Using {len(feature_cols)} features")

X_m = M_train[feature_cols].astype('float32')
y_m = M_train['Win'].astype('int8')
X_w = W_train[feature_cols].astype('float32')
y_w = W_train['Win'].astype('int8')

# Clean up training data (keep stats for test)
del M_train, W_train, M_reg, W_reg, M_tourney, W_tourney
gc.collect()


📊 Building men's training dataset...
   Men: (393646, 17)

📊 Building women's training dataset...
   Women: (281650, 17)

🧩 Using 16 features


13

In [8]:
# =============================================================================
# CELL 7: PRE‑TUNED PARAMETERS (USED WHEN OPTUNA IS OFF)
# =============================================================================
pretuned_lgb_m = {
    'num_leaves': 206, 'learning_rate': 0.0485, 'feature_fraction': 0.972,
    'bagging_fraction': 0.500, 'bagging_freq': 4, 'min_data_in_leaf': 155,
    'lambda_l1': 9.84, 'lambda_l2': 3.23e-7, 'min_gain_to_split': 0.352,
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'verbose': -1, 'num_threads': 4,
    'device': 'gpu' if USE_GPU else 'cpu',
    'gpu_platform_id': 0, 'gpu_device_id': 0
}

pretuned_xgb_m = {
    'learning_rate': 0.01576, 'max_depth': 6, 'min_child_weight': 5.28,
    'subsample': 0.827, 'colsample_bytree': 0.954, 'reg_alpha': 3.81e-5,
    'reg_lambda': 6.72e-6, 'n_estimators': 2000, 'random_state': RANDOM_STATE,
    'n_jobs': 4, 'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'predictor': 'gpu_predictor' if USE_GPU else 'auto'
}

pretuned_cb_m = {
    'iterations': 2000, 'learning_rate': 0.0056, 'depth': 4,
    'l2_leaf_reg': 5.24, 'border_count': 137, 'random_seed': RANDOM_STATE,
    'loss_function': 'Logloss', 'eval_metric': 'AUC', 'verbose': False,
    'task_type': 'GPU' if USE_GPU else 'CPU', 'devices': '0' if USE_GPU else None
}

pretuned_lgb_w = {
    'num_leaves': 506, 'learning_rate': 0.0489, 'feature_fraction': 0.707,
    'bagging_fraction': 0.987, 'bagging_freq': 10, 'min_data_in_leaf': 66,
    'lambda_l1': 0.0808, 'lambda_l2': 3.93e-8, 'min_gain_to_split': 0.924,
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'verbose': -1, 'num_threads': 4,
    'device': 'gpu' if USE_GPU else 'cpu',
    'gpu_platform_id': 0, 'gpu_device_id': 0
}

pretuned_xgb_w = {
    'learning_rate': 0.00921, 'max_depth': 5, 'min_child_weight': 4.77,
    'subsample': 0.942, 'colsample_bytree': 0.842, 'reg_alpha': 0.000468,
    'reg_lambda': 1.56e-6, 'n_estimators': 2000, 'random_state': RANDOM_STATE,
    'n_jobs': 4, 'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'predictor': 'gpu_predictor' if USE_GPU else 'auto'
}

pretuned_cb_w = {
    'iterations': 2000, 'learning_rate': 0.0056, 'depth': 4,
    'l2_leaf_reg': 5.24, 'border_count': 137, 'random_seed': RANDOM_STATE,
    'loss_function': 'Logloss', 'eval_metric': 'AUC', 'verbose': False,
    'task_type': 'GPU' if USE_GPU else 'CPU', 'devices': '0' if USE_GPU else None
}

In [9]:
# =============================================================================
# CELL 8: HYPERPARAMETER OPTIMIZATION FUNCTIONS
# =============================================================================
def optimize_lightgbm(X, y, name):
    print(f"\n⚙️ Optimizing LightGBM for {name}...")
    def objective(trial):
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'num_leaves': trial.suggest_int('num_leaves', 31, 1023),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 300),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
            'verbose': -1,
            'num_threads': 4,
        }
        if USE_GPU:
            params['device'] = 'gpu'
            params['gpu_platform_id'] = 0
            params['gpu_device_id'] = 0
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, valid_sets=[dval],
                              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
            pred = model.predict(X_val)
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best LightGBM params: {study.best_params}")
    return study.best_params

def optimize_xgboost(X, y, name):
    print(f"\n⚙️ Optimizing XGBoost for {name}...")
    def objective(trial):
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'n_estimators': 2000,
            'random_state': RANDOM_STATE,
            'n_jobs': 4,
            'tree_method': 'hist',
            'early_stopping_rounds': 50
        }
        if USE_GPU:
            params['tree_method'] = 'hist'
            params['device'] = 'cuda'
            params['predictor'] = 'gpu_predictor'
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best XGBoost params: {study.best_params}")
    return study.best_params

def optimize_catboost(X, y, name):
    print(f"\n⚙️ Optimizing CatBoost for {name}...")
    def objective(trial):
        params = {
            'iterations': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'loss_function': 'Logloss',
            'eval_metric': 'AUC',
            'random_seed': RANDOM_STATE,
            'verbose': False
        }
        if USE_GPU:
            params['task_type'] = 'GPU'
            params['devices'] = '0'
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best CatBoost params: {study.best_params}")
    return study.best_params

In [10]:
# =============================================================================
# CELL 9: OBTAIN FINAL PARAMETERS (EITHER TUNED OR PRETUNED)
# =============================================================================
if USE_OPTUNA:
    print("\n🔍 Running hyperparameter optimization...")
    # Men
    lgb_params_m = optimize_lightgbm(X_m, y_m, "Men")
    xgb_params_m = optimize_xgboost(X_m, y_m, "Men")
    cb_params_m  = optimize_catboost(X_m, y_m, "Men")
    # Women
    lgb_params_w = optimize_lightgbm(X_w, y_w, "Women")
    xgb_params_w = optimize_xgboost(X_w, y_w, "Women")
    cb_params_w  = optimize_catboost(X_w, y_w, "Women")
else:
    print("\n🔍 Using pre‑tuned parameters (fast mode).")
    lgb_params_m = pretuned_lgb_m
    xgb_params_m = pretuned_xgb_m
    cb_params_m  = pretuned_cb_m
    lgb_params_w = pretuned_lgb_w
    xgb_params_w = pretuned_xgb_w
    cb_params_w  = pretuned_cb_w

[I 2026-03-04 16:46:03,733] A new study created in memory with name: no-name-493b50a7-16fa-4f01-86b6-c9fdb8800020



🔍 Running hyperparameter optimization...

⚙️ Optimizing LightGBM for Men...


  0%|          | 0/20 [00:00<?, ?it/s]

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[45]	valid_0's binary_logloss: 0.515222
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[43]	valid_0's binary_logloss: 0.514367
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[45]	valid_0's binary_logloss: 0.513103
[I 2026-03-04 16:46:29,409] Trial 0 finished with value: 0.5142307848019483 and parameters: {'num_leaves': 449, 'learning_rate': 0.08087864434436551, 'feature_fraction': 0.7506863047721266, 'bagging_fraction': 0.537617962143037, 'bagging_freq': 1, 'min_data_in_leaf': 264, 'lambda_l1': 0.46162590275369986, 'lambda_l2': 3.861541688620009e-06, 'min_gain_to_split': 0.5001491132770479}. Best is trial 0 with value: 0.5142307848019483.
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.52873
Training u

...

[I 2026-03-04 17:34:15,474] Trial 17 finished with value: 0.4658537997853272 and parameters: {'learning_rate': 0.011235054366559923, 'depth': 4, 'l2_leaf_reg': 5.054227710490244, 'border_count': 169}. Best is trial 6 with value: 0.46571175659363306.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-03-04 17:35:06,662] Trial 18 finished with value: 0.46580127292467394 and parameters: {'learning_rate': 0.027480328774639583, 'depth': 8, 'l2_leaf_reg': 8.867572451664063, 'border_count': 214}. Best is trial 6 with value: 0.46571175659363306.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-03-04 17:35:39,042] Trial 19 finished with value: 0.4657832185119221 and parameters: {'learning_rate': 0.043005621805708376, 'depth': 6, 'l2_leaf_reg': 6.4922292986706305, 'border_count': 111}. Best is trial 6 with value: 0.46571175659363306.
   Best CatBoost params: {'learning_rate': 0.022001317960145633, 'depth': 4, 'l2_leaf_reg': 6.768791817511306, 'border_count': 209}


In [11]:
# =============================================================================
# CELL 10: STRATIFIED K-FOLD TRAINING FUNCTION (PRINTS LOG LOSS + ACCURACY)
# =============================================================================
def train_model_cv(X, y, model_name, params, model_constructor):
    """
    Train a model using stratified k-fold, returning fold models and OOF predictions.
    Prints both log loss and accuracy per fold.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_models = []
    oof_pred = np.zeros(len(X))
    cv_scores_log = []
    cv_scores_acc = []
    
    print(f"\n📈 Training {model_name} with {N_FOLDS}-fold CV...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        if model_name == 'LightGBM':
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, valid_sets=[dval],
                              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            pred = model.predict(X_val)
        elif model_name == 'XGBoost':
            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
        elif model_name == 'CatBoost':
            model = cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
        
        fold_score_log = log_loss(y_val, pred)
        fold_score_acc = accuracy_score(y_val, (pred > 0.5).astype(int))
        cv_scores_log.append(fold_score_log)
        cv_scores_acc.append(fold_score_acc)
        oof_pred[val_idx] = pred
        fold_models.append(model)
        print(f"   Fold {fold} LogLoss: {fold_score_log:.5f} | Accuracy: {fold_score_acc:.5f}")
    
    print(f"   CV LogLoss: {np.mean(cv_scores_log):.5f} ± {np.std(cv_scores_log):.5f}")
    print(f"   CV Accuracy: {np.mean(cv_scores_acc):.5f} ± {np.std(cv_scores_acc):.5f}")
    return fold_models, oof_pred

In [12]:
# =============================================================================
# CELL 11: TRAIN ALL MODELS WITH CROSS‑VALIDATION
# =============================================================================
print("\n🚀 Training all models with 5‑fold CV...")

# Men models
lgb_models_m, oof_lgb_m = train_model_cv(X_m, y_m, 'LightGBM', lgb_params_m, lgb.train)
xgb_models_m, oof_xgb_m = train_model_cv(X_m, y_m, 'XGBoost', xgb_params_m, xgb.XGBClassifier)
cb_models_m,  oof_cb_m  = train_model_cv(X_m, y_m, 'CatBoost', cb_params_m, cb.CatBoostClassifier)

# Women models
lgb_models_w, oof_lgb_w = train_model_cv(X_w, y_w, 'LightGBM', lgb_params_w, lgb.train)
xgb_models_w, oof_xgb_w = train_model_cv(X_w, y_w, 'XGBoost', xgb_params_w, xgb.XGBClassifier)
cb_models_w,  oof_cb_w  = train_model_cv(X_w, y_w, 'CatBoost', cb_params_w, cb.CatBoostClassifier)


🚀 Training all models with 5‑fold CV...

📈 Training LightGBM with 5-fold CV...
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[55]	valid_0's l2: 0.171459
   Fold 1 LogLoss: 0.51308 | Accuracy: 0.74293
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[63]	valid_0's l2: 0.171919
   Fold 2 LogLoss: 0.51415 | Accuracy: 0.74434
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[67]	valid_0's l2: 0.170832
   Fold 3 LogLoss: 0.51204 | Accuracy: 0.74525
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[64]	valid_0's l2: 0.171156
   Fold 4 LogLoss: 0.51269 | Accuracy: 0.74411
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[70]	valid_0's l2: 0.170183
   Fold 5 LogLoss: 0.50973 | Accuracy: 0.7

In [13]:
# =============================================================================
# CELL 12: STACKING META‑MODEL (LOGISTIC REGRESSION)
# =============================================================================
print("\n🧠 Training stacking meta‑model...")

# Men
meta_X_m = np.column_stack([oof_lgb_m, oof_xgb_m, oof_cb_m])
meta_model_m = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
meta_model_m.fit(meta_X_m, y_m)
meta_pred_m = meta_model_m.predict_proba(meta_X_m)[:, 1]
meta_loss_m = log_loss(y_m, meta_pred_m)
print(f"   Men stacking LogLoss: {meta_loss_m:.5f}")

# Women
meta_X_w = np.column_stack([oof_lgb_w, oof_xgb_w, oof_cb_w])
meta_model_w = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
meta_model_w.fit(meta_X_w, y_w)
meta_pred_w = meta_model_w.predict_proba(meta_X_w)[:, 1]
meta_loss_w = log_loss(y_w, meta_pred_w)
print(f"   Women stacking LogLoss: {meta_loss_w:.5f}")


🧠 Training stacking meta‑model...
   Men stacking LogLoss: 0.51529
   Women stacking LogLoss: 0.47201


In [14]:
# =============================================================================
# CELL 13: PREPARE TEST DATA (FIXED: seed_cols now matches training)
# =============================================================================
print("\n🧪 Preparing test data...")

sub = sample.copy()
def parse_id(id_):
    season, t1, t2 = id_.split('_')
    return int(season), int(t1), int(t2)

parsed = sub['ID'].apply(parse_id)
parsed = pd.DataFrame(parsed.tolist(), columns=['Season', 'Team1', 'Team2'])

# Merge men's seed features (only numeric seed columns are needed)
men_feat = M_seeds[['Season', 'TeamID'] + ['Seed_Num', 'Seed_Strength', 'Seed_Value', 'Seed_Squared', 'Seed_Percentile']]
parsed = parsed.merge(men_feat.add_prefix('Team1_'),
                      left_on=['Season', 'Team1'], right_on=['Team1_Season', 'Team1_TeamID'],
                      how='left').drop(columns=['Team1_Season', 'Team1_TeamID'])
parsed = parsed.merge(men_feat.add_prefix('Team2_'),
                      left_on=['Season', 'Team2'], right_on=['Team2_Season', 'Team2_TeamID'],
                      how='left').drop(columns=['Team2_Season', 'Team2_TeamID'])

# Merge women's seed features (as fallback) – also numeric only
women_feat = W_seeds[['Season', 'TeamID'] + ['Seed_Num', 'Seed_Strength', 'Seed_Value', 'Seed_Squared', 'Seed_Percentile']]
parsed = parsed.merge(women_feat.add_prefix('WTeam1_'),
                      left_on=['Season', 'Team1'], right_on=['WTeam1_Season', 'WTeam1_TeamID'],
                      how='left').drop(columns=['WTeam1_Season', 'WTeam1_TeamID'])
parsed = parsed.merge(women_feat.add_prefix('WTeam2_'),
                      left_on=['Season', 'Team2'], right_on=['WTeam2_Season', 'WTeam2_TeamID'],
                      how='left').drop(columns=['WTeam2_Season', 'WTeam2_TeamID'])

# Merge men's team stats
stat_cols = ['Avg_pts_scored', 'Avg_pts_allowed', 'Avg_margin', 'Win_pct', 'Games']
parsed = parsed.merge(men_stats.add_prefix('Team1_'),
                      left_on=['Season', 'Team1'], right_on=['Team1_Season', 'Team1_TeamID'],
                      how='left').drop(columns=['Team1_Season', 'Team1_TeamID'])
parsed = parsed.merge(men_stats.add_prefix('Team2_'),
                      left_on=['Season', 'Team2'], right_on=['Team2_Season', 'Team2_TeamID'],
                      how='left').drop(columns=['Team2_Season', 'Team2_TeamID'])

# Merge women's team stats (as fallback)
parsed = parsed.merge(women_stats.add_prefix('WTeam1_'),
                      left_on=['Season', 'Team1'], right_on=['WTeam1_Season', 'WTeam1_TeamID'],
                      how='left').drop(columns=['WTeam1_Season', 'WTeam1_TeamID'])
parsed = parsed.merge(women_stats.add_prefix('WTeam2_'),
                      left_on=['Season', 'Team2'], right_on=['WTeam2_Season', 'WTeam2_TeamID'],
                      how='left').drop(columns=['WTeam2_Season', 'WTeam2_TeamID'])

# Fill missing values (prefer men's, then women's, then defaults)
seed_cols = ['Seed_Num', 'Seed_Strength', 'Seed_Value', 'Seed_Squared', 'Seed_Percentile']
for col in seed_cols:
    m1 = f'Team1_{col}'
    m2 = f'Team2_{col}'
    w1 = f'WTeam1_{col}'
    w2 = f'WTeam2_{col}'
    parsed[m1] = parsed[m1].fillna(parsed[w1])
    parsed[m2] = parsed[m2].fillna(parsed[w2])
    if 'Num' in col:
        parsed[m1] = parsed[m1].fillna(16)
        parsed[m2] = parsed[m2].fillna(16)
    elif 'Strength' in col:
        parsed[m1] = parsed[m1].fillna(1)
        parsed[m2] = parsed[m2].fillna(1)
    else:
        parsed[m1] = parsed[m1].fillna(0)
        parsed[m2] = parsed[m2].fillna(0)

for col in stat_cols:
    m1 = f'Team1_{col}'
    m2 = f'Team2_{col}'
    w1 = f'WTeam1_{col}'
    w2 = f'WTeam2_{col}'
    parsed[m1] = parsed[m1].fillna(parsed[w1])
    parsed[m2] = parsed[m2].fillna(parsed[w2])
    # Default: median of men's stats
    median_val = men_stats[col].median() if col in men_stats.columns else 0
    parsed[m1] = parsed[m1].fillna(median_val)
    parsed[m2] = parsed[m2].fillna(median_val)

# Create test features (mirror training)
# Seed differences
parsed['Seed_Num_Diff'] = parsed['Team1_Seed_Num'] - parsed['Team2_Seed_Num']
parsed['Seed_Strength_Diff'] = parsed['Team1_Seed_Strength'] - parsed['Team2_Seed_Strength']
parsed['Seed_Value_Diff'] = parsed['Team1_Seed_Value'] - parsed['Team2_Seed_Value']
parsed['Seed_Num_Ratio'] = parsed['Team1_Seed_Num'] / (parsed['Team2_Seed_Num'] + 1)
parsed['Seed_Strength_Ratio'] = parsed['Team1_Seed_Strength'] / (parsed['Team2_Seed_Strength'] + 1)

# Team stat differences
parsed['Avg_pts_scored_Diff'] = parsed['Team1_Avg_pts_scored'] - parsed['Team2_Avg_pts_scored']
parsed['Avg_pts_allowed_Diff'] = parsed['Team1_Avg_pts_allowed'] - parsed['Team2_Avg_pts_allowed']
parsed['Avg_margin_Diff'] = parsed['Team1_Avg_margin'] - parsed['Team2_Avg_margin']
parsed['Win_pct_Diff'] = parsed['Team1_Win_pct'] - parsed['Team2_Win_pct']

# Interactions
parsed['Seed_Num_Product'] = parsed['Team1_Seed_Num'] * parsed['Team2_Seed_Num']
parsed['Seed_Sum'] = parsed['Team1_Seed_Num'] + parsed['Team2_Seed_Num']

# Tier indicators
for tier, low, high in [('Elite', 1, 4), ('Contender', 5, 8), ('Mid', 9, 12), ('Low', 13, 16)]:
    parsed[f'Team1_Tier_{tier}'] = ((parsed['Team1_Seed_Num'] >= low) & (parsed['Team1_Seed_Num'] <= high)).astype(int)
    parsed[f'Team2_Tier_{tier}'] = ((parsed['Team2_Seed_Num'] >= low) & (parsed['Team2_Seed_Num'] <= high)).astype(int)

parsed['Same_Tier_Elite'] = ((parsed['Team1_Tier_Elite'] == 1) & (parsed['Team2_Tier_Elite'] == 1)).astype(int)
parsed['Same_Tier_Low'] = ((parsed['Team1_Tier_Low'] == 1) & (parsed['Team2_Tier_Low'] == 1)).astype(int)
parsed['Tier_Gap'] = abs(parsed['Team1_Seed_Num'] // 4 - parsed['Team2_Seed_Num'] // 4)

parsed['Is_Neutral'] = 1
parsed['Round'] = 0  # we don't know tournament round for test games; could impute later

# Keep only feature columns
X_test = parsed[feature_cols].astype('float32')

# Separate men's and women's games based on seed availability
men_mask = parsed['Team1_Seed_Num'].notna() & parsed['Team2_Seed_Num'].notna()
women_mask = ~men_mask
print(f"   Men's games: {men_mask.sum():,} | Women's games: {women_mask.sum():,}")


🧪 Preparing test data...
   Men's games: 519,144 | Women's games: 0


In [15]:
# =============================================================================
# CELL 14: MAKE PREDICTIONS (STACKING ENSEMBLE)
# =============================================================================
print("\n🔮 Generating stacking ensemble predictions...")

def get_base_predictions(models_dict, X_subset):
    """Return a matrix of predictions from each model (averaged over folds)."""
    preds = []
    for name, model_list in models_dict.items():
        fold_pred = np.zeros(len(X_subset))
        for model in model_list:
            if isinstance(model, lgb.Booster):
                fold_pred += model.predict(X_subset) / len(model_list)
            else:
                fold_pred += model.predict_proba(X_subset)[:, 1] / len(model_list)
        preds.append(fold_pred)
    return np.column_stack(preds)

# Men
if men_mask.any():
    base_men = get_base_predictions(
        {'LightGBM': lgb_models_m, 'XGBoost': xgb_models_m, 'CatBoost': cb_models_m},
        X_test[men_mask]
    )
    pred_men = meta_model_m.predict_proba(base_men)[:, 1]
else:
    pred_men = np.array([])

# Women
if women_mask.any():
    base_women = get_base_predictions(
        {'LightGBM': lgb_models_w, 'XGBoost': xgb_models_w, 'CatBoost': cb_models_w},
        X_test[women_mask]
    )
    pred_women = meta_model_w.predict_proba(base_women)[:, 1]
else:
    pred_women = np.array([])

# Fill submission
sub['Pred'] = 0.5
if men_mask.any():
    sub.loc[men_mask, 'Pred'] = pred_men
if women_mask.any():
    sub.loc[women_mask, 'Pred'] = pred_women

# Clip to safe range
sub['Pred'] = np.clip(sub['Pred'], 0.01, 0.99)

print("\n📈 Final prediction stats:")
print(f"   Min: {sub['Pred'].min():.4f} | Max: {sub['Pred'].max():.4f}")
print(f"   Mean: {sub['Pred'].mean():.4f} | Std: {sub['Pred'].std():.4f}")

# Save submission
sub.to_csv("submission.csv", index=False)
print("\n✅ Submission saved as submission.csv")

# Cleanup
gc.collect()
print("\n🏁 Pipeline finished successfully!")


🔮 Generating stacking ensemble predictions...

📈 Final prediction stats:
   Min: 0.0733 | Max: 0.9267
   Mean: 0.4980 | Std: 0.2930

✅ Submission saved as submission.csv

🏁 Pipeline finished successfully!
